# Notebook 04 — LSTM Hourly Forecast (Part 6)

**Prerequisites:** Notebook 01 (hourly data saved).

Builds an LSTM on hourly German load, tunes architecture on a validation window, then recursively forecasts the last 2 years.

### Literature context (cite in your report)

- **Hossain et al. (2020)** — LSTM networks for short-term electrical load forecasting; demonstrates LSTM captures nonlinear temporal dependencies.
- **Nabavi et al. (2024)** — Review of deep learning for electricity load forecasting; LSTM effective for seasonal patterns but needs sufficient data and careful tuning.
- **Hybrid ARIMA-LSTM** papers — combining statistical and neural components can improve accuracy over either alone.

Key point for your discussion: hourly data provides more training samples (~33k train hours) but 2-year recursive forecasts accumulate error. LSTM trades interpretability for flexibility.

In [2]:
!pip install -q pandas numpy matplotlib tensorflow scikit-learn tqdm

from google.colab import drive
drive.mount('/content/drive')

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = "/content/drive/MyDrive/Assignment"
sys.path.insert(0, f"{PROJECT_ROOT}/colab")

from utils import *
from lstm_utils import *

paths = get_paths(PROJECT_ROOT)

hourly = pd.read_csv(paths["processed"] / "hourly_load_mw.csv",
                     parse_dates=["date"], index_col="date")["load_mw"]
hourly = hourly.astype(float).dropna()

train_h, test_h = train_test_split_series(hourly, TEST_HOURS)
print(f"Hourly train: {len(train_h):,} | test: {len(test_h):,}")
print(f"Train: {train_h.index.min()} → {train_h.index.max()}")
print(f"Test:  {test_h.index.min()} → {test_h.index.max()}")

Mounted at /content/drive
Hourly train: 32,928 | test: 17,472
Train: 2015-01-01 00:00:00 → 2018-10-03 23:00:00
Test:  2018-10-04 00:00:00 → 2020-09-30 23:00:00


In [3]:
# Hyperparameter tuning — small grid to avoid Colab timeout
# Evaluate on last 2 weeks of training data (validation window)
VAL_HOURS = 24 * 14
train_fit = train_h.iloc[:-VAL_HOURS]
val_actual = train_h.iloc[-VAL_HOURS:].values

configs = [
    {"seq_len": 168, "units": 64, "n_layers": 1, "lr": 0.001},
    {"seq_len": 168, "units": 32, "n_layers": 1, "lr": 0.001},
    {"seq_len": 24,  "units": 64, "n_layers": 1, "lr": 0.001},
]

tuning_results = []
best_cfg, best_val_rmse = None, np.inf

for cfg in configs:
    print(f"\nTrying: {cfg}")
    model, scaler = train_lstm(
        train_fit.values, seq_len=cfg["seq_len"],
        units=cfg["units"], n_layers=cfg["n_layers"],
        epochs=10, batch_size=64,
    )
    val_preds = recursive_lstm_forecast(
        model, scaler, train_fit.values, VAL_HOURS, cfg["seq_len"],
    )
    val_rmse = float(np.sqrt(np.mean((val_actual - val_preds) ** 2)))
    tuning_results.append({**cfg, "val_RMSE": val_rmse})
    print(f"  Validation RMSE: {val_rmse:.2f} MW")

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_cfg = cfg

    free_memory()

tuning_df = pd.DataFrame(tuning_results).sort_values("val_RMSE")
tuning_df.to_csv(paths["metrics"] / "05_lstm_tuning.csv", index=False)
print("\nBest config:", best_cfg)
tuning_df


Trying: {'seq_len': 168, 'units': 64, 'n_layers': 1, 'lr': 0.001}


LSTM recursive forecast:   0%|          | 0/336 [00:00<?, ?it/s]

  Validation RMSE: 12869.04 MW

Trying: {'seq_len': 168, 'units': 32, 'n_layers': 1, 'lr': 0.001}


LSTM recursive forecast:   0%|          | 0/336 [00:00<?, ?it/s]

  Validation RMSE: 8544.13 MW

Trying: {'seq_len': 24, 'units': 64, 'n_layers': 1, 'lr': 0.001}


LSTM recursive forecast:   0%|          | 0/336 [00:00<?, ?it/s]

  Validation RMSE: 13728.95 MW

Best config: {'seq_len': 168, 'units': 32, 'n_layers': 1, 'lr': 0.001}


,seq_len,units,n_layers,lr,val_RMSE
1,168,32,1,0.001,8544.133871
0,168,64,1,0.001,12869.036068
2,24,64,1,0.001,13728.948027


In [4]:
# Train final LSTM on full training set
SEQ_LEN = best_cfg["seq_len"]
final_model, final_scaler = train_lstm(
    train_h.values,
    seq_len=SEQ_LEN,
    units=best_cfg["units"],
    n_layers=best_cfg["n_layers"],
    epochs=15,
    batch_size=64,
)

# Recursive forecast with checkpoint (resume if interrupted)
ckpt_path = paths["checkpoints"] / "lstm_partial_preds.npy"
partial = np.load(ckpt_path) if ckpt_path.exists() else None

lstm_preds = recursive_lstm_forecast(
    final_model, final_scaler, train_h.values,
    n_steps=len(test_h), seq_len=SEQ_LEN,
    save_every=2000,
    checkpoint=partial,
)

# Save checkpoint after each run (delete this file to start fresh)
np.save(ckpt_path, lstm_preds)
print(f"Forecast complete: {len(lstm_preds):,} hourly predictions")

LSTM recursive forecast:   0%|          | 0/17472 [00:00<?, ?it/s]

Forecast complete: 17,472 hourly predictions


In [5]:
# Evaluation metrics
lstm_metrics = hourly_metrics(
    test_h.values, lstm_preds, train_h.values,
    seasonality=24 * 7,
)
lstm_metrics["model"] = "lstm_hourly"
print(pd.DataFrame([lstm_metrics]).round(3).to_string(index=False))

# Hourly seasonal naive benchmark
sn_hourly = []
hist = train_h.copy()
for i in range(len(test_h)):
    if len(hist) >= 24 * 7:
        sn_hourly.append(hist.iloc[-24 * 7])
    else:
        sn_hourly.append(hist.iloc[-1])
    hist = pd.concat([hist, pd.Series([sn_hourly[-1]], index=[test_h.index[i]])])
sn_hourly = np.array(sn_hourly)

sn_metrics = hourly_metrics(test_h.values, sn_hourly, train_h.values)
sn_metrics["model"] = "seasonal_naive_hourly"
print(pd.DataFrame([sn_metrics]).round(3).to_string(index=False))

lstm_all = pd.DataFrame([sn_metrics, lstm_metrics])
lstm_all.to_csv(paths["metrics"] / "06_lstm_metrics.csv", index=False)

      MAE     RMSE  MASE    Bias       model
11276.558 13769.65 4.886 5190.66 lstm_hourly
     MAE     RMSE  MASE     Bias                 model
5132.542 6735.988 2.224 -293.145 seasonal_naive_hourly


In [6]:
# Plot — first 4 weeks of test for readability (full 2-year plot is very dense)
plot_hours = 24 * 28
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_h.index[:plot_hours], test_h.values[:plot_hours] / 1000,
        label="Actual", color="black", linewidth=1.2)
ax.plot(test_h.index[:plot_hours], lstm_preds[:plot_hours] / 1000,
        label="LSTM", color="crimson", linewidth=1)
ax.plot(test_h.index[:plot_hours], sn_hourly[:plot_hours] / 1000,
        label="Seasonal naive (hourly)", linestyle="--", color="green")
ax.set_title("LSTM hourly forecast — first 4 weeks of test period")
ax.set_ylabel("Load (GW)")
ax.legend()
plt.tight_layout()
plot_and_save(fig, paths["figures"] / "09_lstm_hourly_forecast.png")
plt.show()

# Weekly aggregation for cross-model comparison
test_weekly_actual = (test_h / 1000).resample("W").mean()
test_weekly_lstm = pd.Series(lstm_preds, index=test_h.index).resample("W").mean() / 1000

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_weekly_actual.index, test_weekly_actual, label="Actual (weekly avg)", color="black")
ax.plot(test_weekly_lstm.index, test_weekly_lstm, label="LSTM (weekly avg)", color="crimson")
ax.set_title("LSTM forecast aggregated to weekly averages")
ax.set_ylabel("GW")
ax.legend()
plt.tight_layout()
plot_and_save(fig, paths["figures"] / "10_lstm_weekly_aggregated.png")
plt.show()

pd.DataFrame({
    "actual_mw": test_h.values,
    "lstm_mw": lstm_preds,
}).to_csv(paths["forecasts"] / "lstm_hourly_forecast.csv")

free_memory()
print("\n✓ Notebook 04 complete. All modelling parts done!")
print("Figures →", paths["figures"])
print("Metrics →", paths["metrics"])


✓ Notebook 04 complete. All modelling parts done!
Figures → /content/drive/MyDrive/Assignment/outputs/figures
Metrics → /content/drive/MyDrive/Assignment/outputs/metrics
